In [1]:
import pandas as pd

# Load both files from the raw folder
listings = pd.read_csv("C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\raw\listings.csv", low_memory=False)
reviews = pd.read_csv("C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\raw\reviews.csv")

# How big are they?
print("Listings:", listings.shape)
print("Reviews:", reviews.shape)

# All column names in listings
print(listings.columns.tolist())

# First rows of reviews
print(reviews.head())

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (4135611504.py, line 4)

In [2]:
import pandas as pd

# Folder path (r = raw string, so backslashes are read as normal text)
path = r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\raw"

# Load both files
listings = pd.read_csv(path + r"\listings.csv", low_memory=False)
reviews = pd.read_csv(path + r"\reviews.csv")

# How big are they?
print("Listings:", listings.shape)
print("Reviews:", reviews.shape)

# All column names in listings
print(listings.columns.tolist())

# First rows of reviews
print(reviews.head())

Listings: (10369, 90)
Reviews: (545162, 6)
['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_

In [3]:
# 1) Data types and non-null counts for the key columns
key_cols = ["id", "price", "estimated_revenue_l365d", "estimated_occupancy_l365d",
            "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
            "host_is_superhost", "review_scores_rating", "number_of_reviews", "last_review"]
print(listings[key_cols].info())

# 2) Duplicates and unique ID
print("Full-row duplicates:", listings.duplicated().sum())
print("Is id unique?", listings["id"].is_unique)

# 3) Share of missing values per column (top 20)
missing = (listings.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing.head(20))

# 4) Peek at the price and revenue columns
print(listings[["price", "estimated_revenue_l365d", "estimated_occupancy_l365d"]].head(10))
print(listings["room_type"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 10369 entries, 0 to 10368
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         10369 non-null  int64  
 1   price                      6377 non-null   str    
 2   estimated_revenue_l365d    6377 non-null   float64
 3   estimated_occupancy_l365d  10369 non-null  int64  
 4   neighbourhood_cleansed     10369 non-null  str    
 5   room_type                  10369 non-null  str    
 6   accommodates               10369 non-null  int64  
 7   bedrooms                   9219 non-null   float64
 8   host_is_superhost          10227 non-null  str    
 9   review_scores_rating       9346 non-null   float64
 10  number_of_reviews          10369 non-null  int64  
 11  last_review                9346 non-null   str    
dtypes: float64(3), int64(4), str(5)
memory usage: 972.2 KB
None
Full-row duplicates: 0
Is id unique? True
neighborhoo

In [6]:
# 1) Convert price from text to a number (in a new column, raw price kept)
listings["price_num"] = (listings["price"]
                         .str.replace("$", "", regex=False)
                         .str.replace(",", "", regex=False)
                         .astype(float))
print(listings["price_num"].describe())

# 2) Why is price missing? Compare with availability and occupancy
listings["price_missing"] = listings["price_num"].isna()
print(listings.groupby("price_missing")[["estimated_occupancy_l365d", "availability_365"]].describe().T)
print(listings["has_availability"].value_counts(dropna=False))
print(pd.crosstab(listings["price_missing"], listings["has_availability"]))

# 3) How often is occupancy exactly 255?
print("Occupancy == 255:", (listings["estimated_occupancy_l365d"] == 255).sum())
print("Max occupancy:", listings["estimated_occupancy_l365d"].max())
print("Occupancy == 0:", (listings["estimated_occupancy_l365d"] == 0).sum())

count     6377.000000
mean       346.621932
std        327.297968
min         14.700000
25%        206.670000
50%        291.000000
75%        411.000000
max      11412.000000
Name: price_num, dtype: float64
price_missing                          False        True 
estimated_occupancy_l365d count  6377.000000  3992.000000
                          mean     69.318959    14.738727
                          std      88.278239    32.672979
                          min       0.000000     0.000000
                          25%       8.000000     0.000000
                          50%      31.000000     0.000000
                          75%      86.000000    16.000000
                          max     255.000000   255.000000
availability_365          count  6377.000000  3992.000000
                          mean    148.337620     6.424599
                          std     118.336229    39.590333
                          min       0.000000     0.000000
                          25%      33.

In [7]:
# 1) Verify revenue = price x occupancy for all priced listings
check = listings["price_num"] * listings["estimated_occupancy_l365d"]
diff = (check - listings["estimated_revenue_l365d"]).abs()
print("Max difference:", diff.max())
print("Rows differing by more than 1:", (diff > 1).sum())

# 2) Price outliers
print(listings["price_num"].quantile([0.01, 0.05, 0.5, 0.95, 0.99]))
cols = ["id", "price_num", "room_type", "accommodates", "bedrooms", "number_of_reviews"]
print(listings.nlargest(10, "price_num")[cols])

# 3) Profile the reviews table
print(reviews.dtypes)
print("Missing values:\n", reviews.isna().sum())
print("Duplicate rows:", reviews.duplicated().sum())
print("Is review id unique?", reviews["id"].is_unique)
reviews["date"] = pd.to_datetime(reviews["date"])
print("Date range:", reviews["date"].min(), "to", reviews["date"].max())
print("Orphan reviews:", (~reviews["listing_id"].isin(listings["id"])).sum())
print("Listings with reviews:", reviews["listing_id"].nunique())

Max difference: 0.5
Rows differing by more than 1: 0
0.01      67.3176
0.05     113.5460
0.50     291.0000
0.95     727.5000
0.99    1153.2884
Name: price_num, dtype: float64
                        id  price_num        room_type  accommodates  \
9551   1548025772403046607    11412.0  Entire home/apt             4   
373                2474754    11411.8  Entire home/apt             2   
4364    612289973482163299     7200.0  Entire home/apt            16   
4499    643315499225242586     7200.0  Entire home/apt            16   
3491              42910448     5706.0  Entire home/apt             2   
6267    968203403339639631     3382.6  Entire home/apt             8   
10292  1693616534399481978     3025.0  Entire home/apt             2   
10310  1698438915127575870     2700.0  Entire home/apt             8   
1502              15304003     2477.0  Entire home/apt             4   
6350    984625805593041593     2354.0  Entire home/apt             4   

       bedrooms  number_of_revie

In [8]:
keep_cols = [
    "id", "host_id", "host_is_superhost", "neighbourhood_cleansed", "latitude", "longitude",
    "property_type", "room_type", "accommodates", "bathrooms", "bedrooms", "beds",
    "amenities", "price", "minimum_nights", "maximum_nights",
    "availability_30", "availability_60", "availability_90", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "first_review", "last_review", "review_scores_rating", "review_scores_accuracy",
    "review_scores_cleanliness", "review_scores_checkin", "review_scores_communication",
    "review_scores_location", "review_scores_value", "license",
    "calculated_host_listings_count", "reviews_per_month",
    "estimated_occupancy_l365d", "estimated_revenue_l365d",
]
df = listings[keep_cols].copy()   # work on a copy, raw data untouched
print(df.shape)
print(df["host_is_superhost"].value_counts(dropna=False))   # check values before converting

(10369, 37)
host_is_superhost
f      8513
t      1714
NaN     142
Name: count, dtype: int64


In [9]:
# 1) Superhost: t/f -> 1/0 (unknown stays empty)
df["host_is_superhost"] = df["host_is_superhost"].map({"t": 1, "f": 0})

# 2) Price: text -> number, then drop the text column
df["price_num"] = (df["price"].str.replace("$", "", regex=False)
                              .str.replace(",", "", regex=False)
                              .astype(float))
df = df.drop(columns=["price"])

# 3) Flags (we keep the rows, we just mark them)
df["is_priced"] = df["price_num"].notna()
p99 = df["price_num"].quantile(0.99)
df["price_outlier"] = df["price_num"] > p99
df["occupancy_capped"] = df["estimated_occupancy_l365d"] == 255

# 4) Group tiny room types together
df["room_type_group"] = df["room_type"].replace({"Hotel room": "Other", "Shared room": "Other"})

# 5) Dates: text -> datetime
df["first_review"] = pd.to_datetime(df["first_review"])
df["last_review"] = pd.to_datetime(df["last_review"])

# 6) Simpler names
df = df.rename(columns={"id": "listing_id", "neighbourhood_cleansed": "neighbourhood"})

# 7) Self-check
print(df.shape)
print(df[["is_priced", "price_outlier", "occupancy_capped"]].sum())
print(df["host_is_superhost"].value_counts(dropna=False))
print(df["room_type_group"].value_counts())
print(df.dtypes)

(10369, 41)
is_priced           6377
price_outlier         64
occupancy_capped     886
dtype: int64
host_is_superhost
0.0    8513
1.0    1714
NaN     142
Name: count, dtype: int64
room_type_group
Entire home/apt    8489
Private room       1833
Other                47
Name: count, dtype: int64
listing_id                                 int64
host_id                                    int64
host_is_superhost                        float64
neighbourhood                                str
latitude                                 float64
longitude                                float64
property_type                                str
room_type                                    str
accommodates                               int64
bathrooms                                float64
bedrooms                                 float64
beds                                     float64
amenities                                    str
minimum_nights                           float64
maximum_nights      

In [11]:
# A) Validity checks on listings
print(df[["accommodates", "bedrooms", "bathrooms", "beds", "minimum_nights", "maximum_nights"]].describe())
print("Missing license %:", (df["license"].isna().mean() * 100).round(1))
print("Review rating range:", df["review_scores_rating"].min(), "to", df["review_scores_rating"].max())
print("availability_30 above 30:", (df["availability_30"] > 30).sum())

# B) Clean reviews (work on a copy)
rev = reviews.copy()
rev["date"] = pd.to_datetime(rev["date"])
rev = rev.dropna(subset=["comments"])                     # no text = useless for GenAI
rev["in_listings"] = rev["listing_id"].isin(df["listing_id"])   # flag orphan reviews
rev["review_year"] = rev["date"].dt.year
print(rev.shape)
print(rev["in_listings"].value_counts())
print(rev["review_year"].value_counts().sort_index())

# C) Save the clean files (r = raw string for the Windows path)
out = r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed"
df.to_csv(out + r"\listings_clean.csv", index=False)
rev.to_csv(out + r"\reviews_clean.csv", index=False)
print("Saved")

       accommodates     bedrooms    bathrooms         beds  minimum_nights  \
count  10369.000000  9219.000000  5413.000000  5717.000000    10366.000000   
mean       2.941364     1.634234     1.248014     1.903096        3.591646   
std        1.278998     0.890917     0.529313     1.533525       14.112109   
min        1.000000     0.000000     0.500000     1.000000        1.000000   
25%        2.000000     1.000000     1.000000     1.000000        1.000000   
50%        2.000000     1.000000     1.000000     1.000000        2.000000   
75%        4.000000     2.000000     1.500000     2.000000        3.000000   
max       16.000000    17.000000    17.000000    33.000000      800.000000   

       maximum_nights  
count    1.036600e+04  
mean     4.146884e+05  
std      2.982760e+07  
min      1.000000e+00  
25%      2.100000e+01  
50%      4.000000e+01  
75%      7.287500e+02  
max      2.147484e+09  
Missing license %: 2.2
Review rating range: 1.0 to 5.0
availability_30 above 30: 

In [12]:
# 1) Cap the placeholder: 2,147,483,647 means "no limit"
df["max_nights_clean"] = df["maximum_nights"].clip(upper=365)
df = df.drop(columns=["maximum_nights"])

# 2) Flag very long minimum stays (above 30 nights)
df["long_min_stay"] = df["minimum_nights"] > 30
print("Long minimum stays:", df["long_min_stay"].sum())

# 3) What does the license column contain?
print(df["license"].dropna().head(10).tolist())

# 4) Save again
df.to_csv(out + r"\listings_clean.csv", index=False)
print(df.shape)

Long minimum stays: 48
['0363 607B EA74 0BD8 2F6F', '0363 607B EA74 0BD8 2F6F', '03635399E87602900F47', '0363 E76E F06A C1DD 172C', '0363 4A2B A6AD 0196 F684', '0363 576A D827 5085 6B83', '0363 7F3D 0BAE 28C8 C7D2', '0363 22DC 0E52 B70B 0FB8', '0363 B43C B1D4 2666 3739', '0363 B5A3 5F63 B032 A320']
(10369, 42)


In [13]:
df["has_license"] = df["license"].notna()
df.to_csv(out + r"\listings_clean.csv", index=False)
print(df["has_license"].value_counts())

has_license
True     10146
False      223
Name: count, dtype: int64


In [14]:
print(df.shape)
print(df.columns.tolist())

(10369, 43)
['listing_id', 'host_id', 'host_is_superhost', 'neighbourhood', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'amenities', 'minimum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'license', 'calculated_host_listings_count', 'reviews_per_month', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'price_num', 'is_priced', 'price_outlier', 'occupancy_capped', 'room_type_group', 'max_nights_clean', 'long_min_stay', 'has_license']


In [15]:
# How many comments contain a line break or carriage return?
has_break = rev["comments"].str.contains(r"[\r\n]", regex=True)
print("Comments with line breaks:", has_break.sum())

# Look at the row Snowflake complained about
print(repr(rev.iloc[338547]["comments"][:300]))

Comments with line breaks: 11236
'\\'


In [16]:
for col in ["comments", "reviewer_name"]:
    rev[col] = rev[col].str.replace(r"[\r\n]+", " ", regex=True).str.strip()

# Save again
rev.to_csv(out + r"\reviews_clean.csv", index=False)
print(rev.shape)

(545125, 8)


In [17]:
# How many comments/names contain a backslash?
print("Comments with backslash:", rev["comments"].str.contains("\\", regex=False).sum())

# Replace backslashes with a space in both text columns
for col in ["comments", "reviewer_name"]:
    rev[col] = rev[col].str.replace("\\", " ", regex=False)

# Confirm none are left, then save again
print("Left:", rev["comments"].str.contains("\\", regex=False).sum())
rev.to_csv(out + r"\reviews_clean.csv", index=False)

Comments with backslash: 36
Left: 0


In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv(r"C:\Users\shash\OneDrive\Desktop\Airbnb_Pricing_Analytics\data\processed\listings_clean.csv")
print(df.shape)

ModuleNotFoundError: No module named 'scipy'

: 